# 1-bosqich. Ma'lumotlarning umumiy statistik tahlili

**Maqsad:** ma'lumot bilan tanishish, uning sifatini tekshirish va asosiy ko'rsatkichlarni tavsiflash.
Bu bosqichda model ham, sabab-oqibat haqida xulosa ham yo'q: faqat ma'lumot nimani ko'rsatayotgani tasvirlanadi.

**Tarkib**
1. Dataset pasporti
2. Ustunlar lug'ati
3. Sifat tekshiruvi
4. Asosiy 5 ko'rsatkich (yillar kesimida)
5. Tavsifiy statistika
6. Har bir yil bo'yicha dinamika
7. Oylik ko'rsatkichlar va yillarni solishtirish
8. Mijoz boshiga nisbatlar
9. Taqsimotlar va mavsumiylik
10. Binary belgilar chastotasi
11. Xulosa

In [2]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src import style
from src.data import (load_data, column_dictionary, quality_checks, OUTCOMES, BINARY,
                      WEATHER_NUM, WEEKDAYS_UZ, MONTHS_UZ, SEASONS_UZ, DICTIONARY)

style.setup_notebook()
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
df = load_data()

## 1. Dataset pasporti

In [3]:
groups = pd.Series({c: g.replace(' (target)', '') for c, (g, *_ ) in DICTIONARY.items()}).value_counts()
pd.set_option('display.max_colwidth', 120)
passport = pd.Series({
    "Qatorlar (kunlar)": len(df),
    "Ustunlar (asl faylda)": len(DICTIONARY),
    "Davr": f"{df.date.min():%Y-%m-%d} — {df.date.max():%Y-%m-%d}",
    "Chastota": "kunlik",
    "Yillar bo'yicha kunlar": ", ".join(f"{y}: {n}" for y, n in df.year.value_counts().sort_index().items()),
    "Ustun guruhlari": ", ".join(f"{g}: {n}" for g, n in groups.items()),
}, name="qiymat")
passport.to_frame()

,qiymat
Qatorlar (kunlar),900
Ustunlar (asl faylda),21
Davr,2023-01-01 — 2025-06-18
Chastota,kunlik
Yillar bo'yicha kunlar,"2023: 365, 2024: 366, 2025: 169"
Ustun guruhlari,"Kalendar: 8, Ob-havo: 5, Natija: 4, Tadbir: 3, Sana: 1"


In [4]:
df.drop(columns='year').head()

,date,customers,rice_kg,salad_count,revenue_uzs,temp_c,humidity,wind,rain,snow,...,is_weekend,month,season,is_holiday,ramadan,eid,salary_week,promotion,local_event,football_match
0,2023-01-01,194,44.60,121,10698182,0.40,74.90,5.70,True,False,...,True,1,winter,True,False,False,False,False,False,False
1,2023-01-02,116,27.10,69,6270691,2.80,85.10,1.60,True,False,...,False,1,winter,False,False,False,False,False,False,False
2,2023-01-03,173,40.20,103,9654998,-1.70,89.20,4.40,False,False,...,False,1,winter,False,False,False,True,False,False,False
3,2023-01-04,124,28.50,75,7355468,-3.20,91.30,5.60,False,True,...,False,1,winter,False,False,False,True,False,False,False
4,2023-01-05,171,39.40,94,9742967,-0.50,86.30,5.20,False,False,...,False,1,winter,False,False,False,True,False,False,False


> **Eslatma:** 2025-yil to'liq emas (1-yanvar – 18-iyun, 169 kun, faqat qish va bahor).
> Shuning uchun 2025 ni boshqa yillar bilan **jami** qiymatlar bo'yicha emas, balki **kunlik o'rtacha** va **bir xil oylar** bo'yicha solishtiramiz.

## 2. Ustunlar lug'ati

In [5]:
column_dictionary(df)

,ustun,guruh,tur,birlik,ma'nosi,qiymatlar
0,date,Sana,sana,—,Kuzatuv kuni,2023-01-01 … 2025-06-18
1,customers,Natija,butun son,kishi,Kunlik mijozlar soni,81.0 … 589.0
2,rice_kg,Natija (target),son,kg,Kunlik guruch sarfi,17.6 … 124.5
3,salad_count,Natija,butun son,dona,Sotilgan salatlar soni,45.0 … 367.0
4,revenue_uzs,Natija (target),butun son,so'm,Kunlik tushum,"4,284,917.0 … 33,933,566.0"
5,temp_c,Ob-havo,son,°C,Havo harorati,-7.6 … 42.6
6,humidity,Ob-havo,son,%,Nisbiy namlik,36.0 … 100.0
7,wind,Ob-havo,son,m/s,Shamol tezligi,0.1 … 19.6
8,rain,Ob-havo,binary,—,Yomg'ir yog'dimi,True: 182 kun (20.2%)
9,snow,Ob-havo,binary,—,Qor yog'dimi,True: 59 kun (6.6%)


Ustunlar to'rt guruhga bo'linadi:
- **Natija:** `customers`, `rice_kg`, `salad_count`, `revenue_uzs`. Ular kun oxirida ma'lum bo'ladi. Bashorat qilinadigan targetlar: `rice_kg` va `revenue_uzs`.
- **Ob-havo:** `temp_c`, `humidity`, `wind`, `rain`, `snow`.
- **Kalendar:** hafta kuni, oy, fasl, bayramlar, Ramazon, Hayit, maosh haftasi.
- **Tadbirlar:** aksiya, mahalliy tadbir, futbol o'yini.

## 3. Sifat tekshiruvi

In [6]:
quality_checks(df)

,tekshiruv,natija,holat
0,Bo'sh qiymatlar (NaN),0,✅
1,Takroriy sanalar,0,✅
2,Tushib qolgan kunlar,0,✅
3,weekday sanaga mos emas,0,✅
4,month sanaga mos emas,0,✅
5,is_weekend weekday ga mos emas,0,✅
6,Manfiy natija qiymatlari,0,✅
7,Namlik 0–100% dan tashqarida,0,✅
8,"Qor, lekin harorat > 5°C",0,✅
9,Bir kunda yomg'ir va qor,11,ℹ️ ma'lumot uchun


Ma'lumot toza: bo'sh qiymat, takroriy sana yoki tushib qolgan kun yo'q, kalendar ustunlari sanaga to'liq mos keladi.
11 kunda yomg'ir va qor birga belgilangan. Bu xato emas (qorli yomg'ir bo'lishi mumkin), shuning uchun ma'lumot uchun qayd etildi.

## 4. Asosiy 5 ko'rsatkich

1. **Jami tushum.** 2025 to'liq bo'lmagani uchun kunlik o'rtacha tushum ham beriladi.
2. **O'rtacha kunlik guruch sarfi (kg).**
3. **O'rtacha kunlik mijozlar soni.**
4. **O'rtacha kunlik salat soni.**
5. **Bitta mijozga o'rtacha tushum (so'm).**

In [7]:
def kpis(d):
    return pd.Series({
        "Kunlar soni": len(d),
        "Jami tushum, mln so'm": d.revenue_uzs.sum() / 1e6,
        "Kunlik o'rtacha tushum, mln so'm": d.revenue_uzs.mean() / 1e6,
        "Kunlik o'rtacha guruch, kg": d.rice_kg.mean(),
        "Kunlik o'rtacha mijoz": d.customers.mean(),
        "Kunlik o'rtacha salat": d.salad_count.mean(),
        "Bir mijozga tushum, so'm": d.revenue_uzs.sum() / d.customers.sum(),
    })

kpi = df.groupby("year").apply(kpis).T
kpi["Jami davr"] = kpis(df)
kpi

year,2023,2024,2025,Jami davr
Kunlar soni,365.00,366.00,169.00,900.00
"Jami tushum, mln so'm","4,540.20","4,539.07","1,986.64","11,065.91"
"Kunlik o'rtacha tushum, mln so'm",12.44,12.40,11.76,12.30
"Kunlik o'rtacha guruch, kg",48.86,48.98,45.88,48.35
Kunlik o'rtacha mijoz,222.12,221.69,209.73,219.62
Kunlik o'rtacha salat,132.62,133.38,125.17,131.53
"Bir mijozga tushum, so'm","56,001.38","55,941.26","56,050.00","55,985.42"


In [8]:
# Har bir ko'rsatkich alohida panelda, yillar bo'yicha
items = [("Kunlik o'rtacha tushum, mln so'm", "Tushum, mln so'm/kun"),
         ("Kunlik o'rtacha guruch, kg", "Guruch, kg/kun"),
         ("Kunlik o'rtacha mijoz", "Mijozlar/kun"),
         ("Kunlik o'rtacha salat", "Salat, dona/kun"),
         ("Bir mijozga tushum, so'm", "So'm/mijoz")]
years = [2023, 2024, 2025]
fig = make_subplots(rows=1, cols=5, subplot_titles=[t for _, t in items], horizontal_spacing=0.06)
for i, (row, _) in enumerate(items, start=1):
    vals = kpi.loc[row, years]
    fig.add_bar(x=[str(y) for y in years], y=vals.values, marker_color=[style.YEAR_COLORS[y] for y in years],
                text=[f"{v:,.1f}" if v < 1000 else f"{v:,.0f}" for v in vals.values], textposition="outside",
                showlegend=False, row=1, col=i)
    fig.update_yaxes(range=[0, vals.max() * 1.2], row=1, col=i)
fig.update_layout(title="Asosiy ko'rsatkichlar yillar kesimida (kunlik o'rtacha; 2025 — 1-yanvar–18-iyun)",
                  height=380)
style.save(fig, "01_kpi_by_year")

**Izoh.** 2023 va 2024 deyarli bir xil. 2025 dagi past qiymat o'sish yoki pasayishni anglatmaydi:
2025 da faqat qish va bahor bor, bu fasllarda esa talab tabiiy ravishda past.
Bir mijozga tushum uch yilda ham ~55–56 ming so'm atrofida barqaror.

## 5. Tavsifiy statistika

In [9]:
num_cols = OUTCOMES + WEATHER_NUM
desc = df[num_cols].describe(percentiles=[.25, .5, .75]).T
desc["CV, %"] = df[num_cols].std() / df[num_cols].mean() * 100
desc["skewness"] = df[num_cols].skew()
desc = desc.rename(columns={"count": "n", "mean": "o'rtacha", "std": "std", "min": "min",
                            "25%": "Q1", "50%": "mediana", "75%": "Q3", "max": "max"})
desc

,n,o'rtacha,std,min,Q1,mediana,Q3,max,"CV, %",skewness
customers,900.00,219.62,70.18,81.00,169.75,211.00,260.00,589.00,31.95,0.75
rice_kg,900.00,48.35,15.84,17.60,37.25,46.15,57.40,124.50,32.77,0.83
salad_count,900.00,131.53,42.46,45.00,101.00,127.00,154.00,367.00,32.28,0.80
revenue_uzs,900.00,"12,295,455.28","4,021,770.01","4,284,917.00","9,547,372.75","11,820,943.50","14,588,696.25","33,933,566.00",32.71,0.78
temp_c,900.00,16.12,12.99,-7.60,3.77,15.75,28.20,42.60,80.56,0.08
humidity,900.00,71.05,13.30,36.00,60.80,71.10,80.90,100.00,18.73,-0.04
wind,900.00,3.82,2.77,0.10,1.70,3.30,5.10,19.60,72.63,1.49


**Izoh.**
- Natija ko'rsatkichlarining variatsiya koeffitsienti ~32–33%: kundan kunga o'zgarish sezilarli, ya'ni bashorat qilish uchun "signal" bor.
- Skewness musbat (~0.75–0.85): taqsimot o'ngga qiyshiq, ya'ni kamdan-kam juda gavjum kunlar bor.
  Shu sababli 2-bosqichda logarifmik transformatsiyani ko'rib chiqamiz.
- `wind` ham kuchli o'ngga qiyshiq.

## 6. Har bir yil bo'yicha kunlik dinamika

In [10]:
def yearly_panels(col, title, color, scale=1.0, unit=""):
    fig = make_subplots(rows=3, cols=1, shared_yaxes=True, vertical_spacing=0.08,
                        subplot_titles=[f"{y}" for y in years])
    for i, y in enumerate(years, start=1):
        d = df[df.year == y]
        fig.add_scatter(x=d.date, y=d[col] / scale, mode="lines", line=dict(color=style.GREY, width=0.8),
                        name="kunlik", showlegend=(i == 1), legendgroup="d", row=i, col=1)
        fig.add_scatter(x=d.date, y=(d[col] / scale).rolling(7, center=True).mean(), mode="lines",
                        line=dict(color=color, width=2), name="7 kunlik o'rtacha", showlegend=(i == 1),
                        legendgroup="m", row=i, col=1)
        fig.update_xaxes(range=[f"{y}-01-01", f"{y}-12-31"], dtick="M1", tickformat="%b", row=i, col=1)
        fig.update_yaxes(title_text=unit, row=i, col=1)
    fig.update_layout(title=title, height=720, legend=dict(orientation="h", y=1.06, x=1, xanchor="right"))
    return fig

fig = yearly_panels("rice_kg", "Kunlik guruch sarfi — har bir yil alohida", style.BLUE, unit="kg")
style.save(fig, "01_rice_daily_by_year")

In [11]:
fig = yearly_panels("revenue_uzs", "Kunlik tushum — har bir yil alohida", style.ORANGE, scale=1e6, unit="mln so'm")
style.save(fig, "01_revenue_daily_by_year")

**Izoh.** Har yili bir xil shakl takrorlanadi: qishda past, yozda (iyun–avgust) cho'qqi.
Kundalik grafikdagi "arra" ko'rinishi hafta ichidagi tebranishdan kelib chiqadi (keyingi bo'limlarda ko'rsatiladi).
Alohida keskin cho'qqilar bayram, aksiya yoki tadbir kunlariga to'g'ri keladi.

## 7. Oylik ko'rsatkichlar va yillarni solishtirish

In [12]:
monthly = (df.groupby(["year", "month"])
             .agg(kunlar=("date", "size"), guruch_jami=("rice_kg", "sum"), guruch_ortacha=("rice_kg", "mean"),
                  tushum_jami=("revenue_uzs", "sum"), tushum_ortacha=("revenue_uzs", "mean"))
             .reset_index())

fig = make_subplots(rows=1, cols=2, subplot_titles=["Guruch, kg/kun (oylik o'rtacha)",
                                                    "Tushum, mln so'm/kun (oylik o'rtacha)"])
for y in years:
    d = monthly[monthly.year == y]
    x = [MONTHS_UZ[m - 1] for m in d.month]
    fig.add_bar(x=x, y=d.guruch_ortacha, name=str(y), marker_color=style.YEAR_COLORS[y], legendgroup=str(y), row=1, col=1)
    fig.add_bar(x=x, y=d.tushum_ortacha / 1e6, name=str(y), marker_color=style.YEAR_COLORS[y], legendgroup=str(y),
                showlegend=False, row=1, col=2)
fig.update_layout(barmode="group", title="Oylik o'rtacha ko'rsatkichlar: yillar yonma-yon", height=420)
style.save(fig, "01_monthly_by_year")

In [13]:
# Bir xil oylar (yanvar–iyun) bo'yicha yillarni halol solishtirish
h1 = df[df.month <= 6].groupby("year")[["rice_kg", "revenue_uzs", "customers"]].mean()
h1["revenue_uzs"] /= 1e6
h1.columns = ["Guruch, kg/kun", "Tushum, mln so'm/kun", "Mijoz/kun"]
h1["Guruch: o'tgan yilga nisbatan, %"] = h1["Guruch, kg/kun"].pct_change() * 100
h1.round(2)

,"Guruch, kg/kun","Tushum, mln so'm/kun",Mijoz/kun,"Guruch: o'tgan yilga nisbatan, %"
year,,,,
2023,46.64,11.85,211.67,NaN
2024,46.94,11.84,212.06,0.65
2025,45.88,11.76,209.73,-2.26


**Izoh.** Bir xil davr (yanvar–iyun) bo'yicha solishtirilsa, yillar orasidagi farq 2–3% dan oshmaydi.
Aniq o'sish yoki pasayish trendi kuzatilmaydi. Oylar orasidagi farq esa yillar orasidagi farqdan ancha katta:
talabni asosan **mavsum** belgilaydi.

Mart va aprel oylari yildan-yilga farq qiladi. Sababi Ramazon: u har yili ~11 kun oldinga suriladi
(2023: mart–aprel, 2024: mart–aprel, 2025: to'liq mart).

## 8. Mijoz boshiga nisbatlar

In [14]:
ratios = pd.DataFrame({
    "Guruch / mijoz, kg": df.rice_kg / df.customers,
    "Salat / mijoz, dona": df.salad_count / df.customers,
    "Tushum / mijoz, so'm": df.revenue_uzs / df.customers,
})
r = ratios.describe().T[["mean", "std", "min", "max"]]
r["CV, %"] = ratios.std() / ratios.mean() * 100
r

,mean,std,min,max,"CV, %"
"Guruch / mijoz, kg",0.22,0.01,0.19,0.26,5.08
"Salat / mijoz, dona",0.60,0.03,0.51,0.69,4.96
"Tushum / mijoz, so'm","55,906.74","2,479.50","49,275.84","64,322.63",4.44


In [15]:
corr = df[OUTCOMES].corr()
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Mijozlar va guruch (r = {corr.loc['customers', 'rice_kg']:.3f})",
    f"Mijozlar va tushum (r = {corr.loc['customers', 'revenue_uzs']:.3f})"])
for y in years:
    d = df[df.year == y]
    fig.add_scatter(x=d.customers, y=d.rice_kg, mode="markers", name=str(y), legendgroup=str(y),
                    marker=dict(color=style.YEAR_COLORS[y], size=5, opacity=0.6), row=1, col=1)
    fig.add_scatter(x=d.customers, y=d.revenue_uzs / 1e6, mode="markers", name=str(y), legendgroup=str(y),
                    showlegend=False, marker=dict(color=style.YEAR_COLORS[y], size=5, opacity=0.6), row=1, col=2)
fig.update_xaxes(title_text="Mijozlar soni")
fig.update_yaxes(title_text="Guruch, kg", row=1, col=1)
fig.update_yaxes(title_text="Tushum, mln so'm", row=1, col=2)
fig.update_layout(title="Natija ko'rsatkichlari mijozlar soniga deyarli chiziqli bog'liq", height=420)
style.save(fig, "01_ratios_scatter")

**Izoh.** Bir mijozga guruch (~0.22 kg) va tushum (~55.9 ming so'm) juda barqaror (CV ≈ 4–5%).
Guruch sarfi ham, tushum ham asosan **mijozlar soni** bilan belgilanadi.
Bu ML uchun muhim: guruch va tushumni bevosita bashorat qilish o'rniga, avval mijozlar sonini bashorat qilib,
keyin nisbatga ko'paytirish ham mumkin. Ikkala yondashuv 3-bosqichda solishtiriladi.

## 9. Taqsimotlar va mavsumiylik

In [16]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Guruch sarfi, kg/kun", "Tushum, mln so'm/kun"])
for i, (col, scale, color) in enumerate([("rice_kg", 1, style.BLUE), ("revenue_uzs", 1e6, style.ORANGE)], start=1):
    s = df[col] / scale
    fig.add_histogram(x=s, nbinsx=40, marker_color=color, opacity=0.8, showlegend=False, row=1, col=i)
    for v, dash, label in [(s.mean(), "solid", "o'rtacha"), (s.median(), "dash", "mediana")]:
        fig.add_vline(x=v, line=dict(color="#222", dash=dash, width=1), row=1, col=i,
                      annotation_text=f"{label}: {v:,.1f}", annotation_position="top right" if dash == "solid" else "top left",
                      annotation_font_size=11)
fig.update_yaxes(title_text="Kunlar soni", row=1, col=1)
fig.update_layout(title="Targetlar taqsimoti (o'ngga qiyshiq)", height=380)
style.save(fig, "01_target_hist")

In [17]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Hafta kunlari bo'yicha", "Fasllar bo'yicha"],
                    column_widths=[0.62, 0.38])
for wd in range(7):
    fig.add_box(y=df.loc[df.weekday == wd, "rice_kg"], name=WEEKDAYS_UZ[wd], marker_color=style.BLUE,
                boxmean=True, showlegend=False, row=1, col=1)
for s in ["winter", "spring", "summer", "autumn"]:
    fig.add_box(y=df.loc[df.season == s, "rice_kg"], name=SEASONS_UZ[s], marker_color=style.BLUE,
                boxmean=True, showlegend=False, row=1, col=2)
fig.update_yaxes(title_text="Guruch, kg/kun", row=1, col=1)
fig.update_layout(title="Guruch sarfi: hafta kunlari va fasllar (chiziq — mediana, punktir — o'rtacha)", height=420)
style.save(fig, "01_rice_box_weekday_season")

In [18]:
pivot = df.pivot_table(index="weekday", columns="month", values="rice_kg", aggfunc="mean")
fig = go.Figure(go.Heatmap(z=pivot.values, x=MONTHS_UZ, y=WEEKDAYS_UZ, colorscale=style.SEQ_SCALE,
                           text=np.round(pivot.values, 0), texttemplate="%{text:.0f}", textfont=dict(size=10),
                           colorbar=dict(title="kg")))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="O'rtacha guruch sarfi: hafta kuni × oy (kg/kun)", height=380)
style.save(fig, "01_heatmap_weekday_month")

In [19]:
by_wd = df.groupby("weekday").rice_kg.mean()
by_season = df.groupby("season").rice_kg.mean()
pd.DataFrame({
    "Eng past": [f"{WEEKDAYS_UZ[by_wd.idxmin()]} ({by_wd.min():.1f} kg)", f"{SEASONS_UZ[by_season.idxmin()]} ({by_season.min():.1f} kg)"],
    "Eng yuqori": [f"{WEEKDAYS_UZ[by_wd.idxmax()]} ({by_wd.max():.1f} kg)", f"{SEASONS_UZ[by_season.idxmax()]} ({by_season.max():.1f} kg)"],
    "Farq, marta": [by_wd.max() / by_wd.min(), by_season.max() / by_season.min()],
}, index=["Hafta kuni", "Fasl"]).round(2)

,Eng past,Eng yuqori,"Farq, marta"
Hafta kuni,Dush (38.4 kg),Shan (62.9 kg),1.64
Fasl,Qish (38.6 kg),Yoz (59.2 kg),1.53


**Izoh.** Ikki kuchli mavsumiylik bor:
- **Haftalik:** Dushanbadan Shanbagacha talab o'sib boradi, Yakshanba biroz pasayadi.
- **Yillik:** yozda talab qishdagidan ~1.5 baravar yuqori.

Heatmap bu ikki ta'sir bir-birini kuchaytirishini ko'rsatadi: eng gavjum kunlar yozgi Juma va Shanbalar.

## 10. Binary belgilar chastotasi

In [20]:
names = {"rain": "Yomg'ir", "snow": "Qor", "is_holiday": "Bayram", "ramadan": "Ramazon", "eid": "Hayit",
         "salary_week": "Maosh haftasi", "promotion": "Aksiya", "local_event": "Mahalliy tadbir",
         "football_match": "Futbol o'yini"}
freq = pd.DataFrame({
    "belgi": [names[c] for c in BINARY],
    "kunlar (jami)": [int(df[c].sum()) for c in BINARY],
    "ulushi, %": [df[c].mean() * 100 for c in BINARY],
    **{f"{y}": [int(df.loc[df.year == y, c].sum()) for c in BINARY] for y in years},
}).sort_values("kunlar (jami)", ascending=False).reset_index(drop=True)
freq["ishonchlilik"] = np.where(freq["kunlar (jami)"] < 30, "⚠️ kam (n < 30)", "yetarli")
freq

,belgi,kunlar (jami),"ulushi, %",2023,2024,2025,ishonchlilik
0,Maosh haftasi,296,32.89,120,120,56,yetarli
1,Yomg'ir,182,20.22,63,81,38,yetarli
2,Ramazon,90,10.00,30,30,30,yetarli
3,Aksiya,66,7.33,25,27,14,yetarli
4,Qor,59,6.56,24,26,9,yetarli
5,Mahalliy tadbir,45,5.00,17,19,9,yetarli
6,Futbol o'yini,44,4.89,13,22,9,yetarli
7,Bayram,18,2.00,7,7,4,⚠️ kam (n < 30)
8,Hayit,6,0.67,2,2,2,⚠️ kam (n < 30)


In [21]:
f = freq.sort_values("kunlar (jami)")
fig = go.Figure(go.Bar(x=f["kunlar (jami)"], y=f["belgi"], orientation="h",
                       marker_color=np.where(f["kunlar (jami)"] < 30, style.RED, style.BLUE),
                       text=[f"{n} ({p:.1f}%)" for n, p in zip(f["kunlar (jami)"], f["ulushi, %"])],
                       textposition="outside"))
fig.add_vline(x=30, line=dict(color=style.GREY, dash="dash"), annotation_text="n = 30", annotation_position="top")
fig.update_layout(title="Binary belgilar necha kun uchraydi (qizil — kam namuna)", height=400,
                  xaxis=dict(title="Kunlar soni", range=[0, f["kunlar (jami)"].max() * 1.2]))
style.save(fig, "01_binary_freq")

**Izoh.** Hayit (6 kun) va bayram (18 kun) juda kam uchraydi. Ularning ta'siri haqidagi har qanday xulosa
katta noaniqlikka ega bo'ladi, shuning uchun 2-bosqichda ishonch oraliqlari bilan baholanadi.

## 11. Xulosa

1. **Ma'lumot toza va izchil:** 900 kun (2023-01-01 — 2025-06-18), bo'sh qiymat va uzilish yo'q.
   2025-yil to'liq emas, shuning uchun yillar faqat bir xil oylar bo'yicha solishtiriladi.
2. **Yillar orasida aniq trend yo'q.** Yanvar–iyun davri bo'yicha 2023, 2024 va 2025 bir-biriga yaqin.
3. **Kuchli ikki mavsumiylik bor:** haftalik (Dushanba eng past, Shanba eng yuqori) va yillik (yoz ≈ qish × 1.5).
4. **Guruch va tushum mijozlar soniga deyarli chiziqli bog'liq** (r ≈ 0.99). Mijoz boshiga nisbatlar barqaror (CV ≈ 4–5%).
5. **Targetlar o'ngga qiyshiq.** Log transformatsiya ko'rib chiqiladi.
6. **Ba'zi belgilar kam uchraydi** (Hayit: 6 kun, bayram: 18 kun). Ularning ta'sirini ehtiyotkorlik bilan baholash kerak.

**2-bosqichda:** qaysi omillar guruch va tushumga qanchalik ta'sir qilishini, omillarning o'zaro
bog'liqligini (korrelyatsiya, kovariatsiya, multikollinearlik) tekshiramiz va modelga kiradigan featurelarni tanlaymiz.